[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astudentfromsustech/agents-course/blob/main/practice/p2_vit_mnist/vit_mnist.ipynb)

# p2 · A transformer that reads handwritten digits

An image is not a sequence. A **Vision Transformer** makes it one: cut the 28×28 image
into 7×7 patches, flatten each into a vector, and call that the sequence — 16 tokens,
plus a `[CLS]` token whose final vector the classifier reads. Everything after that is
chapter 01 unchanged: Q/K/V, softmax, multi-head, residual, norm.

Three things the chapter *claims*, measured here instead of asserted:

| | what it shows |
|---|---|
| `train` | it works, and the attention maps are readable |
| `position` | attention is permutation-equivariant — strip the positional encoding and shuffling the patches changes **nothing** |
| `scale` | a CNN's locality is a free prior, worth most when data is scarce |

Runs on a Mac GPU (MPS), a Colab GPU, or plain CPU — the next cell picks whichever it finds.

In [ ]:
# Setup. Works on Colab and on your own machine; no torchvision needed.
import os, sys, importlib, urllib.request

URL = ("https://raw.githubusercontent.com/astudentfromsustech/agents-course"
       "/main/practice/p2_vit_mnist/vit_mnist.py")
IN_COLAB = "google.colab" in sys.modules

# On Colab always re-fetch, so a restarted session cannot keep running stale code.
# Locally never overwrite: that file is yours to edit.
if IN_COLAB or not os.path.exists("vit_mnist.py"):
    urllib.request.urlretrieve(URL, "vit_mnist.py")

import vit_mnist as V
importlib.reload(V)      # a warm kernel holds the OLD module even after a fresh download

print("device:", V.DEV)  # mps = your Mac's GPU | cuda = Colab GPU | cpu = fallback
print("has show_data:", hasattr(V, "show_data"))

## The data, and what a patch is

MNIST arrives as one 11 MB `.npz` — 60 000 training images, 10 000 test.
`PatchEmbed` is a `Conv2d` whose kernel equals its stride, so it does **no mixing at
all**: it only reshapes pixels into tokens. Every bit of spatial reasoning after that
point is done by attention.

In [ ]:
d = V.load()
(Xtr, Ytr), (Xte, Yte) = d
print("train", tuple(Xtr.shape), " test", tuple(Xte.shape))

m = V.ViT().to(V.DEV)
print("tokens per image:", V.NP, "patches +1 CLS =", V.NP + 1)
print("params:", f"{sum(p.numel() for p in m.parameters()):,}")
print("one image -> patch sequence:", tuple(m.patch(Xte[:1]).shape))   # [1, 16, 64]

# ...and actually look at it
from IPython.display import Image, display
display(Image(V.show_data(d)))

**Read the bottom strip.** That row of 16 squares is the entire input — the model never
receives the 4x4 arrangement, only those tokens in order. Token 5 and 6 hold the top bar of
the `7`, 10 and 13 the diagonal.

Which is the setup for experiment 2: if you shuffled that strip, would the model notice?

## 1 · Train it

~11 s on an M-series Mac. Watch the test accuracy climb.

In [ ]:
m = V.exp_train(d)

`attention_maps.png` is written next to the notebook. The bottom row is the `[CLS]`
token's attention over the 4×4 patch grid — **not uniform**: on a `1` the mass runs up the
stroke, on a `0` it sits around the ring.

Attention weights are one of the few interpretable things in a neural network, because
they are a distribution over *inputs* rather than over hidden features.

In [ ]:
from IPython.display import Image, display
display(Image("attention_maps.png"))

## 2 · Position: the one thing attention cannot see

Train two identical models, one with the positional encoding switched off. Then **shuffle
the patch order at test time**.

Predict the four numbers before you run it.

In [ ]:
rows = V.exp_position(d)

The `+0.00` is the whole point. The model with no positional encoding is not merely
*robust* to the shuffle — it is **mathematically incapable of noticing it**, because the
only thing it ever received was a set. That is permutation equivariance, measured.

The gap in the first column is what position buys you.

## 3 · Scale: what a prior is worth

Same parameter budget, same number of gradient steps, growing training set. A CNN has
locality *hard-coded*; the transformer has to learn it.

In [ ]:
out = V.exp_scale(d)

**Read the gap column, not the accuracy columns.** The CNN's locality prior is free
knowledge handed over before training starts, and it is worth about **19 points at
N = 1 000 and 3 points at N = 60 000**.

No crossover here — the ViT never overtakes the CNN on MNIST, because all of MNIST sits
inside the small-data regime (the crossover in the ViT paper is somewhere past 100 M
images). What a laptop can show you is the **slope**: the prior paying off less and less.
That slope is the whole argument of the chapter's `Self-attention vs CNN` section.

## Things worth trying

- `V.PATCH = 4` (49 tokens) or `14` (4 tokens) — where does accuracy break, and is the cost
  compute or information?
- `V.LAYERS = 1` — how much does one round of mixing lose?
- Shuffle the **pixels inside** each patch instead of the patches. Why does that hurt the
  no-position model, when shuffling patches did not?
- `PatchEmbed` is a `Conv2d` with `kernel_size == stride`. Make them differ so patches
  overlap — you have just reintroduced a locality prior. Does the N = 1 000 number move
  toward the CNN's?